# Range — code-only review companion

This notebook documents the chapter's causal input/target contract. It contains no data, simulator, weights, historical plots or embedded experimental results.

Private execution is disabled by default. Read `../docs/CHAPTER_ALIGNMENT.md` before interpreting a new score as historical evidence. Install the package with neural/notebooks extras first. In Colab connected to a local runtime, run the installation in the active kernel and use paths visible inside that container.


In [ ]:
from pathlib import Path
import sys

# Install from the clone using the current kernel, if necessary:
# %pip install -e /path/to/radar-tracklet-thesis[neural,notebooks]
print("Kernel:", sys.executable)
COORDINATE = "range"
ARCHITECTURE = "GRU-CNN"  # explicit family, NOT a claim of historical tuning reproduction
RUN_PRIVATE_WINDOW = False
ALLOW_NEW_TRAINING = False
PRIVATE_DATA_ROOT = None  # Path outside this repository
PRIVATE_SPLIT_MANIFEST = None  # original private trajectory split
PRIVATE_WINDOW_MODEL = None  # trusted compatible .keras checkpoint, or explicit new training
PRIVATE_WINDOW_SCALER_X = None  # original persisted scaler for checkpoint inference
PRIVATE_WINDOW_SCALER_Y = None
EPOCHS = 100  # new-run ceiling, not the historical best epoch


## Canonical windows

Inputs 0–24 predict clean target 25. Both scalers fit training only. Clean values construct targets and metrics, not predictor features. No teacher forcing. New training is opt-in; no file is overwritten or saved by this example. Keep outputs private and label new runs separately.


In [ ]:
if RUN_PRIVATE_WINDOW:
    import tensorflow as tf
    from radar_tracklets.models import build_window_model, LastTimeStep
    from radar_tracklets.workflows import prepare_windows
    from radar_tracklets.evaluation import reconstruct, pooled_metrics
    if PRIVATE_DATA_ROOT is None or PRIVATE_SPLIT_MANIFEST is None:
        raise ValueError("Supply authorized private data and the original split manifest")
    persisted_scalers = None
    if not ALLOW_NEW_TRAINING:
        import joblib
        if PRIVATE_WINDOW_SCALER_X is None or PRIVATE_WINDOW_SCALER_Y is None:
            raise ValueError("Existing checkpoints require their original persisted scalers")
        persisted_scalers = (joblib.load(PRIVATE_WINDOW_SCALER_X), joblib.load(PRIVATE_WINDOW_SCALER_Y))
    data, scaler_X, scaler_y = prepare_windows(
        PRIVATE_DATA_ROOT, PRIVATE_SPLIT_MANIFEST, COORDINATE, fitted_scalers=persisted_scalers)
    train, validation, test = (data[k] for k in ("train_files", "val_files", "test_files"))
    if ALLOW_NEW_TRAINING:
        if PRIVATE_WINDOW_MODEL is not None:
            raise ValueError("Choose new training OR an existing model, not both")
        model = build_window_model(COORDINATE, ARCHITECTURE, seed=42)
        history = model.fit(
            train["X"], train["y"], validation_data=(validation["X"], validation["y"]),
            epochs=EPOCHS, batch_size=64, shuffle=True,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
                       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=.5, min_lr=1e-6)])
    elif PRIVATE_WINDOW_MODEL is not None:
        # Requires scalers/data protocol matching the supplied model. No compatibility
        # with historical tuned models is presumed merely from matching dimensions.
        model = tf.keras.models.load_model(
            PRIVATE_WINDOW_MODEL, compile=False,
            custom_objects={"LastTimeStep": LastTimeStep, "DebrisTracking>LastTimeStep": LastTimeStep})
    else:
        raise ValueError("Supply a compatible private model or authorize new training")
    prediction_scaled = model.predict(test["X"], batch_size=64)
    prediction = reconstruct(COORDINATE, prediction_scaled, scaler_y,
                             baseline=test.get("baseline"), pivot=test.get("pivot"))
    print(pooled_metrics(prediction, test["clean"], COORDINATE))
    print("Trajectories:", len(set(test["trajectory_id"])), "targets:", len(test["clean"]))
else:
    print("Private-data window execution disabled; no training or inference performed.")
